In [11]:
from pathlib import Path

page_path = Path("streamlit_app/pages/1_Team_Tournament.py")

team_tournament_code = """
import streamlit as st

from utils.data_loader import load_all_data
from utils.filters import get_tournament_options, get_team_options, filter_team
from utils.style import apply_global_style
from utils.cards import metric_card, text_card
from utils.visuals import (
    plot_most_frequent_xi_shape,
    plot_team_shot_map,
    plot_team_heatmap
)

st.set_page_config(
    page_title="Team Tournament",
    layout="wide"
)

apply_global_style()

data = load_all_data()

matches = data["matches"]
events = data["events"]
team_master = data["team_master"]
player_master = data["player_master"]
most_frequent_xi = data["most_frequent_xi"]
highlighted_matches = data["highlighted_matches"]
team_similarity = data["team_similarity"]
assets = data["assets"]

st.sidebar.title("Filters")

tournament_id = st.sidebar.selectbox(
    "Tournament",
    get_tournament_options(matches)
)

team = st.sidebar.selectbox(
    "Team",
    get_team_options(team_master, tournament_id)
)

team_row = filter_team(team_master, tournament_id, team)

asset_row = assets[
    assets["team"] == team
]

if len(asset_row) > 0:
    flag_path = asset_row.iloc[0]["flag_path"]
    primary_color = asset_row.iloc[0]["primary_color"]
else:
    flag_path = None
    primary_color = "#111827"

# ============================================================
# HEADER
# ============================================================

st.markdown("<div class='hero-card'>", unsafe_allow_html=True)

col_flag, col_title = st.columns([1, 6])

with col_flag:
    if flag_path:
        st.image(flag_path, width=90)

with col_title:
    st.markdown(
        f"<h1 style='color:{primary_color}; margin-bottom:0;'>{team}</h1>",
        unsafe_allow_html=True
    )
    st.markdown(
        f"<div class='page-subtitle'>{tournament_id} · Team Tournament Analysis</div>",
        unsafe_allow_html=True
    )

st.markdown("</div>", unsafe_allow_html=True)

# ============================================================
# KPI ROW
# ============================================================

if len(team_row) > 0:

    row = team_row.iloc[0]

    c1, c2, c3, c4, c5 = st.columns(5)

    with c1:
        metric_card("Matches", int(row.get("matches", 0)))

    with c2:
        metric_card("Goals", round(row.get("goals", 0), 2))

    with c3:
        metric_card("xG", round(row.get("xg", 0), 2))

    with c4:
        metric_card("Attack Score", round(row.get("attack_score", 0), 1))

    with c5:
        metric_card("Style", row.get("general_style", "N/A"))

# ============================================================
# TABS
# ============================================================

tab_overview, tab_players, tab_visuals, tab_matches, tab_similarity = st.tabs([
    "Overview",
    "Players",
    "Visuals",
    "Highlighted Matches",
    "Similar Teams"
])

# ============================================================
# OVERVIEW
# ============================================================

with tab_overview:

    c1, c2 = st.columns([1.1, 1])

    with c1:
        st.subheader("Most Frequent XI")

        fig = plot_most_frequent_xi_shape(
            most_frequent_xi=most_frequent_xi,
            tournament_id=tournament_id,
            team=team
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No XI visual available.")

    with c2:
        st.subheader("Team Identity")

        if len(team_row) > 0:
            row = team_row.iloc[0]

            text_card(
                "General Style",
                row.get("general_style", "N/A")
            )

            text_card(
                "Offensive Style",
                row.get("offensive_style", "N/A")
            )

            text_card(
                "Defensive Style",
                row.get("defensive_style", "N/A")
            )

# ============================================================
# PLAYERS
# ============================================================

with tab_players:

    st.subheader("Squad Overview")

    squad = player_master[
        (player_master["tournament_id"] == tournament_id) &
        (player_master["team"] == team)
    ].sort_values("importance_score", ascending=False)

    top_players = squad.head(5)

    cols = st.columns(5)

    for i, (_, p) in enumerate(top_players.iterrows()):
        with cols[i]:
            metric_card(
                p["player"].split()[-1],
                round(p.get("importance_score", 0), 1),
                p.get("general_role", "")
            )

    st.markdown("### Full Squad")

    display_cols = [
        "player",
        "position",
        "general_role",
        "importance_score",
        "importance_tier",
        "xg",
        "shots",
        "passes",
        "pressures",
        "strengths",
        "weaknesses"
    ]

    existing_cols = [
        c for c in display_cols
        if c in squad.columns
    ]

    st.dataframe(
        squad[existing_cols],
        use_container_width=True,
        hide_index=True
    )

# ============================================================
# VISUALS
# ============================================================

with tab_visuals:

    c1, c2 = st.columns(2)

    with c1:
        st.subheader("Shot Map")

        fig = plot_team_shot_map(
            events=events,
            tournament_id=tournament_id,
            team=team
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No shots available.")

    with c2:
        st.subheader("Activity Heatmap")

        fig = plot_team_heatmap(
            events=events,
            tournament_id=tournament_id,
            team=team
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)
        else:
            st.info("No event locations available.")

# ============================================================
# HIGHLIGHTED MATCHES
# ============================================================

with tab_matches:

    st.subheader("Highlighted Team Matches")

    hm = highlighted_matches[
        (highlighted_matches["tournament_id"] == tournament_id) &
        (highlighted_matches["team"] == team) &
        (highlighted_matches["level"] == "team")
    ].sort_values("match_score_rank")

    display_cols = [
        "match_label",
        "xg",
        "shots",
        "passes",
        "pressures",
        "recoveries",
        "match_score",
        "match_score_rank"
    ]

    existing_cols = [
        c for c in display_cols
        if c in hm.columns
    ]

    st.dataframe(
        hm[existing_cols],
        use_container_width=True,
        hide_index=True
    )

# ============================================================
# SIMILAR TEAMS
# ============================================================

with tab_similarity:

    st.subheader("Similar Teams")

    sim = team_similarity[
        (team_similarity["tournament_id"] == tournament_id) &
        (team_similarity["team"] == team)
    ].sort_values("similarity_rank")

    display_cols = [
        "similar_team",
        "similarity_score",
        "similarity_rank"
    ]

    existing_cols = [
        c for c in display_cols
        if c in sim.columns
    ]

    st.dataframe(
        sim[existing_cols],
        use_container_width=True,
        hide_index=True
    )
"""

page_path.write_text(team_tournament_code.strip())

print("1_Team_Tournament.py updated successfully!")

1_Team_Tournament.py updated successfully!


In [12]:
from pathlib import Path

page_path = Path("streamlit_app/pages/4_Player_Tournament.py")

player_tournament_code = """
import streamlit as st

from utils.data_loader import load_all_data
from utils.filters import (
    get_tournament_options,
    get_team_options,
    get_player_options,
    filter_player
)

from utils.style import apply_global_style
from utils.cards import metric_card, text_card

from utils.visuals import (
    plot_player_heatmap
)

st.set_page_config(
    page_title="Player Tournament",
    layout="wide"
)

apply_global_style()

data = load_all_data()

matches = data["matches"]
events = data["events"]

player_master = data["player_master"]
player_role_fit = data["player_role_fit"]
player_similarity = data["player_similarity"]
highlighted_matches = data["highlighted_matches"]
assets = data["assets"]

st.sidebar.title("Filters")

tournament_id = st.sidebar.selectbox(
    "Tournament",
    get_tournament_options(matches)
)

team = st.sidebar.selectbox(
    "Team",
    get_team_options(player_master, tournament_id)
)

player = st.sidebar.selectbox(
    "Player",
    get_player_options(player_master, tournament_id, team)
)

profile = filter_player(
    player_master,
    tournament_id,
    team,
    player
)

asset_row = assets[
    assets["team"] == team
]

if len(asset_row) > 0:
    flag_path = asset_row.iloc[0]["flag_path"]
    primary_color = asset_row.iloc[0]["primary_color"]
else:
    flag_path = None
    primary_color = "#111827"

# ============================================================
# HEADER
# ============================================================

st.markdown("<div class='hero-card'>", unsafe_allow_html=True)

c1, c2 = st.columns([1, 6])

with c1:
    if flag_path:
        st.image(flag_path, width=85)

with c2:
    st.markdown(
        f"<h1 style='color:{primary_color}; margin-bottom:0;'>{player}</h1>",
        unsafe_allow_html=True
    )

    st.markdown(
        f"<div class='page-subtitle'>{team} · {tournament_id}</div>",
        unsafe_allow_html=True
    )

st.markdown("</div>", unsafe_allow_html=True)

# ============================================================
# KPI ROW
# ============================================================

if len(profile) > 0:

    row = profile.iloc[0]

    c1, c2, c3, c4, c5 = st.columns(5)

    with c1:
        metric_card(
            "Matches",
            int(row.get("matches", 0))
        )

    with c2:
        metric_card(
            "Role",
            row.get("general_role", "N/A")
        )

    with c3:
        metric_card(
            "Importance",
            round(row.get("importance_score", 0), 1)
        )

    with c4:
        metric_card(
            "xG",
            round(row.get("xg", 0), 2)
        )

    with c5:
        metric_card(
            "Shots",
            round(row.get("shots", 0), 2)
        )

# ============================================================
# TABS
# ============================================================

tab1, tab2, tab3, tab4 = st.tabs([
    "Overview",
    "Visuals",
    "Role Fit",
    "Similar Players"
])

# ============================================================
# OVERVIEW
# ============================================================

with tab1:

    c1, c2 = st.columns([1, 1])

    with c1:

        if len(profile) > 0:

            row = profile.iloc[0]

            text_card(
                "General Role",
                row.get("general_role", "N/A")
            )

            text_card(
                "Strengths",
                row.get("strengths", "No strengths available.")
            )

            text_card(
                "Weaknesses",
                row.get("weaknesses", "No weaknesses available.")
            )

    with c2:

        st.subheader("Player Heatmap")

        fig = plot_player_heatmap(
            events=events,
            tournament_id=tournament_id,
            team=team,
            player=player
        )

        if fig is not None:
            st.pyplot(fig, use_container_width=True)

# ============================================================
# VISUALS
# ============================================================

with tab2:

    st.subheader("Highlighted Matches")

    hm = highlighted_matches[
        (highlighted_matches["tournament_id"] == tournament_id) &
        (highlighted_matches["team"] == team) &
        (highlighted_matches["player"] == player)
    ]

    st.dataframe(
        hm,
        use_container_width=True,
        hide_index=True
    )

# ============================================================
# ROLE FIT
# ============================================================

with tab3:

    role_fit = player_role_fit[
        (player_role_fit["tournament_id"] == tournament_id) &
        (player_role_fit["team"] == team) &
        (player_role_fit["player"] == player)
    ].sort_values(
        "role_fit_score",
        ascending=False
    )

    st.dataframe(
        role_fit,
        use_container_width=True,
        hide_index=True
    )

# ============================================================
# SIMILAR PLAYERS
# ============================================================

with tab4:

    sim = player_similarity[
        (player_similarity["tournament_id"] == tournament_id) &
        (player_similarity["team"] == team) &
        (player_similarity["player"] == player)
    ].sort_values(
        "similarity_rank"
    )

    display_cols = [
        "similar_player",
        "similar_team",
        "similarity_score",
        "similarity_rank"
    ]

    existing_cols = [
        c for c in display_cols
        if c in sim.columns
    ]

    st.dataframe(
        sim[existing_cols],
        use_container_width=True,
        hide_index=True
    )
"""

page_path.write_text(player_tournament_code.strip())

print("4_Player_Tournament.py updated!")

4_Player_Tournament.py updated!


In [13]:
page_path = Path("streamlit_app/pages/3_Team_Comparison.py")

team_comparison_code = """
import streamlit as st

from utils.data_loader import load_all_data
from utils.filters import (
    get_tournament_options,
    get_team_options
)

from utils.style import apply_global_style
from utils.cards import metric_card

st.set_page_config(
    page_title="Team Comparison",
    layout="wide"
)

apply_global_style()

data = load_all_data()

matches = data["matches"]
team_master = data["team_master"]

st.sidebar.title("Filters")

tournament_id = st.sidebar.selectbox(
    "Tournament",
    get_tournament_options(matches)
)

teams = get_team_options(
    team_master,
    tournament_id
)

team_a = st.sidebar.selectbox(
    "Team A",
    teams,
    index=0
)

team_b = st.sidebar.selectbox(
    "Team B",
    teams,
    index=1 if len(teams) > 1 else 0
)

profile_a = team_master[
    (team_master["tournament_id"] == tournament_id) &
    (team_master["team"] == team_a)
]

profile_b = team_master[
    (team_master["tournament_id"] == tournament_id) &
    (team_master["team"] == team_b)
]

st.title("Team Comparison")
st.caption(tournament_id)

c1, c2 = st.columns(2)

with c1:

    st.markdown(f"## {team_a}")

    if len(profile_a) > 0:

        row = profile_a.iloc[0]

        metric_card("Goals", round(row.get("goals", 0), 2))
        metric_card("xG", round(row.get("xg", 0), 2))
        metric_card("Style", row.get("general_style", "N/A"))

with c2:

    st.markdown(f"## {team_b}")

    if len(profile_b) > 0:

        row = profile_b.iloc[0]

        metric_card("Goals", round(row.get("goals", 0), 2))
        metric_card("xG", round(row.get("xg", 0), 2))
        metric_card("Style", row.get("general_style", "N/A"))
"""

page_path.write_text(team_comparison_code.strip())

print("3_Team_Comparison.py updated!")

3_Team_Comparison.py updated!


In [14]:
page_path = Path("streamlit_app/pages/6_Player_Comparison.py")

player_comparison_code = """
import streamlit as st

from utils.data_loader import load_all_data
from utils.filters import get_tournament_options

from utils.style import apply_global_style
from utils.cards import metric_card

st.set_page_config(
    page_title="Player Comparison",
    layout="wide"
)

apply_global_style()

data = load_all_data()

matches = data["matches"]
player_master = data["player_master"]

st.sidebar.title("Filters")

tournament_id = st.sidebar.selectbox(
    "Tournament",
    get_tournament_options(matches)
)

players = sorted(
    player_master[
        player_master["tournament_id"] == tournament_id
    ]["player"].dropna().unique()
)

player_a = st.sidebar.selectbox(
    "Player A",
    players,
    index=0
)

player_b = st.sidebar.selectbox(
    "Player B",
    players,
    index=1 if len(players) > 1 else 0
)

profile_a = player_master[
    (player_master["tournament_id"] == tournament_id) &
    (player_master["player"] == player_a)
]

profile_b = player_master[
    (player_master["tournament_id"] == tournament_id) &
    (player_master["player"] == player_b)
]

st.title("Player Comparison")
st.caption(tournament_id)

c1, c2 = st.columns(2)

with c1:

    st.markdown(f"## {player_a}")

    if len(profile_a) > 0:

        row = profile_a.iloc[0]

        metric_card("Role", row.get("general_role", "N/A"))
        metric_card("xG", round(row.get("xg", 0), 2))
        metric_card("Importance", round(row.get("importance_score", 0), 1))

with c2:

    st.markdown(f"## {player_b}")

    if len(profile_b) > 0:

        row = profile_b.iloc[0]

        metric_card("Role", row.get("general_role", "N/A"))
        metric_card("xG", round(row.get("xg", 0), 2))
        metric_card("Importance", round(row.get("importance_score", 0), 1))
"""

page_path.write_text(player_comparison_code.strip())

print("6_Player_Comparison.py updated!")

6_Player_Comparison.py updated!
